# Technique-Level Multiclass Classification of Prompt Injection Attacks

Notebook eksperimen untuk klasifikasi **6 kelas** berdasarkan teknik dan vektor serangan yang terlihat pada teks prompt. Notebook ini mengikuti `Panduan Penyusunan.md` (3 November 2026).

**Status:** rev4, 8 November 2026: output ke `results_rev4/` dan sel 15B membuat Gambar 2–4 paper langsung dari hasil run (nilai tidak berubah). Sebelumnya rev3 (8 Nov: versi pustaka dikunci, analisis tambahan paper diekspor ke CSV, info runtime dicatat). Revisi 6 November 2026 (perbaikan audit) tetap berlaku. Semua output lama dihapus karena kode berubah. Jalankan **Runtime → Restart session and run all** dari awal sebelum mengambil angka untuk paper.

**Perubahan revisi 6 Nov:**
1. Regex normalisasi whitespace diperbaiki (`r"\s+"`); cache embedding kini diberi sidik jari teks sehingga otomatis dihitung ulang bila teks berubah.
2. SVC RBF (R3) kini ikut memakai `class_weight="balanced"`, konsisten dengan LinearSVC dan RF.
3. Hard negative diperbaiki: regex pemicu sebelumnya tidak pernah cocok, dan subset kini mencakup prompt benign dari semua sumber.
4. Tabel komposisi kelas per kategori asli (cek isi K5) dan rincian recall K5.
5. Analisis ketahanan: metrik pada subset test non-augmentasi.
6. Bootstrap CI untuk ke-12 konfigurasi (resampel berpasangan) + CI F1 per kelas model terbaik.
7. AUC biner untuk SVM memakai `decision_function`; waktu inferensi R3 kini juga dilaporkan termasuk waktu encoding.
8. Interpretasi ComplementNB memakai bobot terpusat antar-kelas; confusion matrix dan heatmap recall kategori disimpan ke file.
9. `save_table` kini menyimpan index bertingkat (sebelumnya label algoritma/representasi hilang dari `final_results_mean_std.csv`).
10. **K5 diganti nama menjadi _Code Execution, Indirect and Agent Injection_** (keputusan tim 6 Nov). Pemetaan kategori tidak berubah; nama baru diterapkan di sel pemetaan.
11. Sel duplikat (sensitivitas K2/HarmBench) dihapus; urutan bagian dan nama file di panduan interpretasi diselaraskan.

**Sumber data:** `neuralchemy/Prompt-injection-dataset`, konfigurasi `core`, split resmi train/validation/test. Hanya kolom `text` yang menjadi fitur model. `category` dan `source` hanya untuk pemetaan/audit analisis; kolom metadata lain tidak digunakan sebagai fitur.

## 00. Ringkasan rancangan penelitian

- Target: K0–K5 (6 kelas), dengan nama:
  - K0 — Benign
  - K1 — Instruction Override
  - K2 — Role-play and Conversational Jailbreak
  - K3 — Obfuscation and Token Smuggling
  - K4 — Prompt and Information Extraction
  - K5 — Code Execution, Indirect and Agent Injection
- Representasi: R1 TF-IDF kata (1–2 gram), R2 TF-IDF karakter (`char_wb`, 3–5 gram), R3 sentence embedding 384 dimensi.
- Model: Naive Bayes, SVM, Random Forest, dan MLP (4 × 3 = 12 konfigurasi inti).
- Metrik utama: Macro-F1 6 kelas. Pendukung: accuracy, balanced accuracy, weighted-F1, MCC, F1 per kelas, FPR benign, FNR serangan, CI bootstrap, dan McNemar.
- `edge_case` dan `control` dikeluarkan; jumlahnya harus dilaporkan per split.
- Analisis dibatasi pada teknik/vektor yang tampak di teks; tidak memakai kerangka dampak atau uji Bahasa Indonesia.

**Catatan taksonomi:** K1–K3 terutama merupakan teknik; K4 dan K5 lebih dekat ke intent/vektor. Ini adalah taksonomi operasional berdasarkan kategori dataset dan pemetaan ke HackAPrompt, bukan ontologi universal. Satu prompt dapat memuat beberapa teknik, tetapi dataset memberi satu label; keterbatasan multilabel dicatat.

## 01. Environment dan reproducibility

Jalankan pada Google Colab. Paket dipasang tanpa memaksa upgrade seluruh environment. Embedding R3 disimpan sebagai `.npy` agar tidak dihitung ulang pada run berikutnya.

In [ ]:
# Versi dikunci sama dengan requirements.txt run 6 Nov agar hasil dapat direproduksi.
# numpy/pandas/scipy memakai bawaan Colab (mengganti numpy di Colab memaksa restart).
!pip -q install "scikit-learn==1.6.1" "sentence-transformers==5.7.0" "datasets==4.8.5" "statsmodels==0.15.0"


In [ ]:
import os, re, sys, json, time, random, warnings, platform, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
SEEDS_FINAL = [42, 43, 44, 45, 46]
CLASS_CODES = [f"K{i}" for i in range(6)]
CLASS_IDS = list(range(6))
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

# Google Drive untuk mapping.csv, cache embedding, log, tabel, dan gambar.
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/prompt_injection_project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_PATH = PROJECT_DIR / "mapping.csv"
CACHE_DIR = PROJECT_DIR / "embeddings"
RESULTS_DIR = PROJECT_DIR / "results_rev4"   # run rev3 tersimpan di results_rev3/
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR = RESULTS_DIR / "models"
INPUTS_DIR = PROJECT_DIR / "inputs"   # label_audit_wildguard.csv
for folder in [CACHE_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def detect_runtime():
    """Catat perangkat keras runtime untuk dilaporkan di paper (kolom ms/1.000)."""
    info = {"platform": platform.platform(), "processor": platform.processor(),
            "cpu_count": os.cpu_count(),
            "colab_tpu": bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"))}
    try:
        import torch
        info["cuda_available"] = torch.cuda.is_available()
        info["cuda_device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        info["cuda_available"] = None
    try:
        info["cpu_model"] = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo")
                                 if l.startswith("model name"))
    except Exception:
        info["cpu_model"] = None
    return info

RUNTIME_INFO = detect_runtime()
print("Runtime:", RUNTIME_INFO)
print("Python:", sys.version)
print("Project directory:", PROJECT_DIR)

In [ ]:
def save_figure(filename, dpi=300):
    """Simpan figure aktif ke results/figures."""
    output_path = FIGURES_DIR / filename

    plt.savefig(
        output_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Figure saved: {output_path}")
    return output_path


def save_table(dataframe, filename):
    """Simpan DataFrame ke CSV."""
    if dataframe is None:
        return None

    output_path = TABLES_DIR / filename

    dataframe.to_csv(
        output_path,
        index=any(name is not None for name in dataframe.index.names),
        encoding="utf-8-sig"
    )

    print(f"Table saved: {output_path}")
    return output_path

## 02. Dataset loading

Dataset menggunakan konfigurasi `core`. Split resmi terdiri dari 4.391 train, 941 validation, dan 942 test sebelum pengecualian. Dataset card menyatakan pemisahan dilakukan berdasarkan `group_id` sehingga grup tidak tersebar antar-split; klaim ini perlu dirujuk ke dataset card pada paper.

In [ ]:
from datasets import load_dataset

DATASET_ID = "neuralchemy/Prompt-injection-dataset"
dataset = load_dataset(DATASET_ID, "core")

df_train_raw = dataset["train"].to_pandas()
df_val_raw = dataset["validation"].to_pandas()
df_test_raw = dataset["test"].to_pandas()

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(df_train_raw), len(df_val_raw), len(df_test_raw)]
})
display(split_summary)
print("Columns:", df_train_raw.columns.tolist())
assert [len(df_train_raw), len(df_val_raw), len(df_test_raw)] == [4391, 941, 942], (
    "Ukuran split berbeda dari panduan; periksa versi/config dataset sebelum melanjutkan."
)

## 03. Exploratory Data Analysis

EDA melaporkan distribusi kategori asli, sumber, label biner bawaan, panjang teks, dan data hilang.

In [ ]:
df_all_raw = pd.concat(
    [df_train_raw.assign(split="train"),
     df_val_raw.assign(split="validation"),
     df_test_raw.assign(split="test")],
    ignore_index=True
)

print("Missing values by split:")
display(pd.DataFrame({
    "train": df_train_raw.isna().sum(),
    "validation": df_val_raw.isna().sum(),
    "test": df_test_raw.isna().sum()
}))

category_counts = (
    df_train_raw["category"].astype(str).value_counts()
    .rename_axis("category").reset_index(name="train_count")
)
print("Distinct category values in train:", df_train_raw["category"].nunique())
display(category_counts)

category_split = pd.crosstab(df_all_raw["category"], df_all_raw["split"])
display(category_split.reindex(columns=["train", "validation", "test"], fill_value=0))

source_category = pd.crosstab(df_train_raw["source"], df_train_raw["category"])
display(source_category)

df_all_raw["text_length"] = df_all_raw["text"].fillna("").astype(str).str.len()
display(df_all_raw.groupby("split")["text_length"].agg(["count","min","median","mean","max"]))

plt.figure(figsize=(12, 7))
sns.barplot(data=category_counts, x="train_count", y="category")
plt.title("Original Category Counts in Train")
plt.xlabel("Rows"); plt.ylabel("Original category"); plt.tight_layout(); plt.show()

plt.figure(figsize=(11, 6))
sns.heatmap(source_category, cmap="Blues")
plt.title("Original Category by Source (Train)")
plt.tight_layout(); plt.show()

## 04. Pemetaan kategori ke enam kelas

Pemetaan dibaca dari `mapping.csv`, bukan dictionary pemetaan yang ditanam di pipeline. File pendamping `mapping.csv` harus ditempatkan di folder Google Drive `prompt_injection_project`.

Kategori bertanda **PROVISIONAL** wajib ditinjau: `adversarial`, `token_injection`, `response_manipulation`, `output_manipulation`, dan `chain_of_thought`. Notebook mencetak 10 contoh per kategori agar dua anggota dapat memeriksa dan mengunci pemetaan sebelum eksperimen final. Jangan menganggap mapping provisional telah tervalidasi.

**Keputusan 6 Nov 2026:**
- `adversarial` berisi payload eksekusi kode yang diketik langsung (mis. `os.system('whoami')`) dan membentuk sekitar 86% K5. Kategori ini **tetap di K5**, dan K5 **diganti nama** dari *Indirect and Agent Injection* menjadi ***Code Execution, Indirect and Agent Injection*** agar nama kelas sesuai isinya. Nama baru diterapkan lewat `CLASS_NAME_OVERRIDES` di sel berikut, jadi `mapping.csv` lama tetap bisa dipakai. Tabel komposisi di 04.1 dan rincian recall K5 di bagian 10 menjadi dasar pelaporannya di paper.
- `output_manipulation` bercampur: sebagian contoh adalah ekstraksi instruksi/system prompt (lebih dekat ke K4).
- Nama kelas dibaca dari `mapping.csv`, jadi penggantian nama cukup dilakukan di file tersebut.

In [ ]:
EXCLUDED_CATEGORIES = {
    "edge_case": "Kategori tidak terdefinisi secara operasional untuk enam kelas teknik.",
    "control": "Baris kontrol berlabel serangan tetapi teksnya aman dan tidak memuat teknik serangan."
}

if not MAPPING_PATH.exists():
    raise FileNotFoundError(
        f"File {MAPPING_PATH} belum ditemukan. Unggah mapping.csv pendamping ke folder ini."
    )

mapping_df = pd.read_csv(MAPPING_PATH).fillna("")

# Keputusan tim 6 Nov 2026: K5 diganti nama agar sesuai isinya (didominasi payload
# eksekusi kode dari kategori `adversarial`). Pemetaan kategori -> kelas tidak berubah.
CLASS_NAME_OVERRIDES = {"K5": "Code Execution, Indirect and Agent Injection"}
REASON_OVERRIDES = {
    "adversarial": "Payload eksekusi kode yang diketik langsung; tetap di K5 "
                   "setelah K5 diganti nama (keputusan 6 Nov 2026)."
}
for code_, name_ in CLASS_NAME_OVERRIDES.items():
    mapping_df.loc[mapping_df["class_code"] == code_, "class_name"] = name_
for category_, reason_ in REASON_OVERRIDES.items():
    assert category_ in set(mapping_df["category"]), category_
    mapping_df.loc[mapping_df["category"] == category_, "reason"] = reason_
required_mapping_columns = {"category", "class_code", "class_name", "reason"}
assert required_mapping_columns.issubset(mapping_df.columns)
assert mapping_df["category"].is_unique, "Setiap kategori asli hanya boleh muncul sekali."
assert set(mapping_df["class_code"]).issubset(set(CLASS_CODES))
assert set(mapping_df["class_code"]) == set(CLASS_CODES)

class_names = (
    mapping_df.drop_duplicates("class_code")
    .set_index("class_code")["class_name"].to_dict()
)
assert len(class_names) == 6
assert class_names["K0"] == "Benign"
assert class_names["K5"] == CLASS_NAME_OVERRIDES["K5"]

observed_categories = set(df_all_raw["category"].dropna().astype(str).unique())
mapped_categories = set(mapping_df["category"].astype(str))
excluded_categories = set(EXCLUDED_CATEGORIES)
unmapped_categories = observed_categories - mapped_categories - excluded_categories
unused_mapping_categories = mapped_categories - observed_categories

print("Observed category values:", len(observed_categories))
print("Mapped categories:", len(mapped_categories))
print("Excluded categories:", sorted(excluded_categories))
print("Unmapped categories:", sorted(unmapped_categories))
print("Mapping entries absent from this dataset version:", sorted(unused_mapping_categories))
assert not unmapped_categories, "Ada kategori dataset yang belum dipetakan atau dikecualikan."
assert not (mapped_categories & excluded_categories)

display(mapping_df.sort_values(["class_code", "category"]))

provisional_mask = mapping_df["reason"].str.contains("PROVISIONAL", case=False, na=False)
provisional_categories = mapping_df.loc[provisional_mask, "category"].tolist()
print("Kategori yang wajib ditinjau:", provisional_categories)

for category in provisional_categories:
    print(f"\n===== {category}: 10 contoh =====")
    examples = df_train_raw.loc[
        df_train_raw["category"].astype(str).eq(category), "text"
    ].dropna().head(10)
    for idx, example in enumerate(examples, start=1):
        safe_example = str(example).replace("\n", " ")
        print(f"{idx}. {safe_example[:350]}{' ...' if len(safe_example) > 350 else ''}")

print("Setelah tinjauan dua anggota, edit mapping.csv (class_code, class_name, reason), "
      "hapus tanda PROVISIONAL bila sudah disepakati, lalu jalankan notebook dari awal.")

### 04.1 Audit eksklusi dan distribusi kelas

Target jumlah eksklusi dari panduan adalah 59 / 16 / 14 untuk train / validation / test (total 89). Setelah eksklusi, target ukuran data adalah 4.332 / 925 / 928. Angka aktual dihitung ulang dari dataset yang dimuat dan ditampilkan; assertion akan menghentikan proses bila berbeda.

In [ ]:
def apply_category_mapping(frame):
    out = frame.copy()
    out["category"] = out["category"].astype(str)
    out["class_code"] = out["category"].map(
        mapping_df.set_index("category")["class_code"]
    )
    out = out.loc[out["class_code"].notna()].copy()
    out["class_id"] = out["class_code"].str.replace("K", "", regex=False).astype(int)
    out["class_name"] = out["class_code"].map(class_names)
    return out.reset_index(drop=True)

train = apply_category_mapping(df_train_raw)
val = apply_category_mapping(df_val_raw)
test = apply_category_mapping(df_test_raw)

exclusion_rows = []
for split_name, raw in [
    ("train", df_train_raw), ("validation", df_val_raw), ("test", df_test_raw)
]:
    counts = raw["category"].astype(str).value_counts()
    for category, reason in EXCLUDED_CATEGORIES.items():
        exclusion_rows.append({
            "split": split_name, "category": category,
            "count": int(counts.get(category, 0)), "reason": reason
        })
exclusion_audit_df = pd.DataFrame(exclusion_rows)
display(exclusion_audit_df)

excluded_by_split = (
    exclusion_audit_df.groupby("split")["count"].sum()
    .reindex(["train", "validation", "test"])
)
retained_by_split = pd.Series({
    "train": len(train), "validation": len(val), "test": len(test)
})
print("Excluded:", excluded_by_split.to_dict())
print("Retained:", retained_by_split.to_dict())

assert excluded_by_split.tolist() == [59, 16, 14], (
    "Jumlah eksklusi berbeda dari target panduan; periksa split/mapping."
)
assert retained_by_split.tolist() == [4332, 925, 928]

for frame in [train, val, test]:
    assert frame["class_id"].between(0, 5).all()
    assert frame["class_name"].notna().all()
    assert frame["text"].notna().all()

class_dist_df = pd.DataFrame({
    "train": train["class_id"].value_counts().reindex(CLASS_IDS, fill_value=0),
    "validation": val["class_id"].value_counts().reindex(CLASS_IDS, fill_value=0),
    "test": test["class_id"].value_counts().reindex(CLASS_IDS, fill_value=0)
})
class_dist_df.insert(0, "class_code", CLASS_CODES)
class_dist_df.insert(1, "class_name", [class_names[f"K{i}"] for i in CLASS_IDS])
class_dist_df["total"] = class_dist_df[["train", "validation", "test"]].sum(axis=1)
display(class_dist_df)

# Komposisi tiap kelas menurut kategori asli (train+validation+test).
composition_df = (
    pd.concat([train, val, test], ignore_index=True)
    .groupby(["class_code", "class_name", "category"]).size()
    .rename("rows").reset_index()
)
composition_df["share_in_class"] = (
    composition_df["rows"] / composition_df.groupby("class_code")["rows"].transform("sum")
)
composition_df = composition_df.sort_values(["class_code", "rows"], ascending=[True, False])
display(composition_df)
dominant = composition_df.loc[composition_df.groupby("class_code")["rows"].idxmax()]
for _, row in dominant.iterrows():
    print(f"{row['class_code']}: kategori terbesar '{row['category']}' "
          f"({row['share_in_class']:.0%} dari kelas)")

print("Category values in train:", df_train_raw["category"].nunique(),
      "(dataset card menyebut 29 kategori serangan; config core memuat 31 nilai termasuk benign, edge_case, control).")

## 05. Preprocessing dan pemeriksaan fitur

Normalisasi hanya merapikan whitespace. Stopword, tanda baca, simbol, Unicode, string base64, dan kapitalisasi tidak dibuang. Lowercase hanya diterapkan oleh TF-IDF kata; TF-IDF karakter mempertahankan kapitalisasi. Fitur model harus berasal **hanya dari kolom `text`**.

In [ ]:
def preprocess_text(value):
    return re.sub(r"\s+", " ", str(value)).strip()

for frame in [train, val, test]:
    frame["text_clean"] = frame["text"].map(preprocess_text)

X_train = train["text_clean"].astype(str).tolist()
X_val = val["text_clean"].astype(str).tolist()
X_test = test["text_clean"].astype(str).tolist()
y_train = train["class_id"].to_numpy(dtype=int)
y_val = val["class_id"].to_numpy(dtype=int)
y_test = test["class_id"].to_numpy(dtype=int)

assert all(text != "" for text in X_train + X_val + X_test)
assert "text_clean" in train.columns
FEATURE_SOURCE = "text"
assert FEATURE_SOURCE == "text"

preprocessing_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(val), len(test)],
    "empty_text": [(frame["text_clean"] == "").sum() for frame in [train, val, test]],
    "changed_by_whitespace_normalization": [
        (frame["text"].astype(str) != frame["text_clean"]).sum()
        for frame in [train, val, test]
    ]
})
display(preprocessing_summary)

## 06. Representasi teks

- **R1 Word TF-IDF:** unigram–bigram; `sublinear_tf=True`, `min_df=2`, lowercase, maksimum 20.000 fitur.
- **R2 Character TF-IDF:** `char_wb`, n-gram 3–5; `sublinear_tf=True`, `min_df=2`, tanpa lowercase, maksimum 20.000 fitur.
- **R3 Sentence embedding:** `sentence-transformers/all-MiniLM-L12-v2`, 384 dimensi, teks mentah tanpa preprocessing tambahan. Embedding setiap split dihitung satu kali dan disimpan sebagai `.npy`; nama file memuat sidik jari (hash) teks sehingga cache lama tidak terpakai bila preprocessing atau mapping berubah.

R1/R2 menangkap jejak leksikal dan bentuk karakter; R3 menjadi pembanding semantik.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TFIDF_CONFIG = {
    "R1": dict(analyzer="word", ngram_range=(1, 2), sublinear_tf=True,
               min_df=2, lowercase=True, max_features=20000),
    "R2": dict(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True,
               min_df=2, lowercase=False, max_features=20000)
}
REPRESENTATIONS = ["R1", "R2", "R3"]

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

import hashlib

def texts_fingerprint(texts):
    """Hash isi dan urutan teks; cache hanya dipakai ulang bila teksnya identik."""
    digest = hashlib.sha256()
    for text in texts:
        digest.update(text.encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()[:12]

EMBEDDING_FINGERPRINTS = {}

def load_or_create_embeddings(split_name, texts):
    fingerprint = texts_fingerprint(texts)
    EMBEDDING_FINGERPRINTS[split_name] = fingerprint
    cache_path = CACHE_DIR / f"{split_name}_all-MiniLM-L12-v2_{fingerprint}.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape != (len(texts), 384):
            raise ValueError(f"Cache {cache_path} tidak cocok dengan jumlah data/dimensi.")
        return arr
    arr = embedding_model.encode(
        texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=False
    ).astype(np.float32)
    assert arr.shape == (len(texts), 384)
    np.save(cache_path, arr)
    return arr

X_train_R3 = load_or_create_embeddings("train", X_train)
X_val_R3 = load_or_create_embeddings("validation", X_val)
X_test_R3 = load_or_create_embeddings("test", X_test)
print(X_train_R3.shape, X_val_R3.shape, X_test_R3.shape)

## 07. Algoritma, grid, dan protokol tuning

Empat keluarga model menghasilkan 12 konfigurasi inti (4 × 3). Varian NB dan SVM berbeda pada R3 karena embedding dense dapat bernilai negatif: R1/R2 menggunakan ComplementNB dan LinearSVC; R3 menggunakan GaussianNB dan SVC RBF.

- SVM (LinearSVC dan SVC RBF) serta Random Forest memakai `class_weight="balanced"` pada konfigurasi utama. NB dan MLP tidak memiliki parameter ini.
- MLP langsung menerima TF-IDF sparse pada R1/R2. Untuk R3, MLP didahului StandardScaler.
- Tuning dilakukan pada train/validation melalui `PredefinedSplit`; konfigurasi terbaik dipilih dengan Macro-F1 validation. Model final di-fit ulang pada train+validation dan test dipakai sekali.

In [ ]:
from sklearn.model_selection import PredefinedSplit, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import ComplementNB, MultinomialNB, GaussianNB
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_auc_score
)
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
import scipy.sparse as sp

PARAM_GRIDS = {
    "Naive Bayes": {
        "R1": {"alpha": [0.01, 0.1, 0.5, 1.0]},
        "R2": {"alpha": [0.01, 0.1, 0.5, 1.0]},
        "R3": {"var_smoothing": [1e-9, 1e-8, 1e-7]}
    },
    "SVM": {
        "R1": {"C": [0.1, 0.5, 1, 2, 5]},
        "R2": {"C": [0.1, 0.5, 1, 2, 5]},
        "R3": {"C": [0.5, 1, 5], "gamma": ["scale", 0.01, 0.1]}
    },
    "Random Forest": {
        rep: {"max_depth": [None, 30], "min_samples_leaf": [1, 2]}
        for rep in REPRESENTATIONS
    },
    "MLP": {
        rep: {"hidden_layer_sizes": [(128,), (256,)], "alpha": [1e-4, 1e-3]}
        for rep in REPRESENTATIONS
    }
}

def build_classifier(algorithm, representation, params, seed=SEED,
                     use_class_weight=True):
    if algorithm == "Naive Bayes":
        if representation in ("R1", "R2"):
            return ComplementNB(**params)
        return GaussianNB(**params)
    if algorithm == "SVM":
        if representation in ("R1", "R2"):
            return LinearSVC(
                **params, class_weight="balanced" if use_class_weight else None,
                random_state=seed, max_iter=20000
            )
        return SVC(
            **params, kernel="rbf", decision_function_shape="ovr",
            class_weight="balanced" if use_class_weight else None
        )
    if algorithm == "Random Forest":
        return RandomForestClassifier(
            **params, n_estimators=300, class_weight="balanced" if use_class_weight else None,
            random_state=seed, n_jobs=-1
        )
    if algorithm == "MLP":
        return MLPClassifier(
            **params, activation="relu", solver="adam",
            early_stopping=True, max_iter=300,
            n_iter_no_change=15, random_state=seed
        )
    raise ValueError(algorithm)

def make_model(algorithm, representation, params, seed=SEED,
               use_class_weight=True):
    clf = build_classifier(algorithm, representation, params, seed, use_class_weight)
    if representation in ("R1", "R2"):
        return Pipeline([
            ("tfidf", TfidfVectorizer(**TFIDF_CONFIG[representation])),
            ("classifier", clf)
        ])
    steps = []
    if algorithm == "MLP":
        steps.append(("scale", StandardScaler()))
    steps.append(("classifier", clf))
    return Pipeline(steps)

def representation_data(rep):
    if rep in ("R1", "R2"):
        return X_train, X_val, X_test
    return X_train_R3, X_val_R3, X_test_R3

def fit_score_candidate(algorithm, representation, params, seed=SEED,
                        use_class_weight=True):
    Xtr, Xva, _ = representation_data(representation)
    X_combined = list(Xtr) + list(Xva) if representation in ("R1", "R2") else np.vstack([Xtr, Xva])
    y_combined = np.concatenate([y_train, y_val])
    test_fold = np.concatenate([
        np.full(len(y_train), -1, dtype=int),
        np.zeros(len(y_val), dtype=int)
    ])
    predefined = PredefinedSplit(test_fold=test_fold)
    # Fit hanya pada train fold dan skor pada validation fold.
    model = make_model(algorithm, representation, params, seed, use_class_weight)
    train_idx, val_idx = next(predefined.split())
    X_fit = [X_combined[i] for i in train_idx] if isinstance(X_combined, list) else X_combined[train_idx]
    X_hold = [X_combined[i] for i in val_idx] if isinstance(X_combined, list) else X_combined[val_idx]
    model.fit(X_fit, y_combined[train_idx])
    pred = model.predict(X_hold)
    return float(f1_score(y_combined[val_idx], pred, average="macro", zero_division=0)), model

print("Core model/representation configurations:", 4 * 3)

### 07.1 Hyperparameter tuning pada validation

Semua kandidat dievaluasi pada validation dengan Macro-F1. Test tidak digunakan untuk memilih hyperparameter. `class_weight` utama diaktifkan untuk SVM (LinearSVC dan SVC) dan RF; MLP/NB tidak memiliki parameter tersebut.

In [ ]:
tuning_rows = []
best_configs = {}

for algorithm in ["Naive Bayes", "SVM", "Random Forest", "MLP"]:
    for representation in REPRESENTATIONS:
        candidates = list(ParameterGrid(PARAM_GRIDS[algorithm][representation]))
        best_score, best_params = -np.inf, None
        for params in candidates:
            start = time.perf_counter()
            score, _ = fit_score_candidate(
                algorithm, representation, params, seed=SEED,
                use_class_weight=True
            )
            elapsed = time.perf_counter() - start
            tuning_rows.append({
                "algorithm": algorithm, "representation": representation,
                "params": params.copy(), "validation_macro_f1": score,
                "tuning_seconds": elapsed
            })
            if score > best_score:
                best_score, best_params = score, params.copy()
        best_configs[(algorithm, representation)] = {
            "params": best_params, "validation_macro_f1": best_score
        }

tuning_df = pd.DataFrame(tuning_rows)
best_config_df = pd.DataFrame([
    {"algorithm": a, "representation": r,
     "params": str(v["params"]), "validation_macro_f1": v["validation_macro_f1"]}
    for (a, r), v in best_configs.items()
]).sort_values("validation_macro_f1", ascending=False)
display(tuning_df.sort_values("validation_macro_f1", ascending=False))
display(best_config_df)
assert len(best_configs) == 12

### 07.2 Refit train+validation dan evaluasi test satu kali

Konfigurasi ditetapkan dari validation. Model final kemudian dilatih ulang pada gabungan train+validation. RF dan MLP menggunakan lima seed; NB dan SVM satu seed. Waktu inferensi dilaporkan dalam milidetik per 1.000 prompt. Untuk R1/R2 angka ini sudah mencakup transformasi TF-IDF. Untuk R3, kolom `inference_ms_per_1000` hanya mencakup classifier; kolom `inference_ms_per_1000_incl_features` menambahkan waktu encoding embedding test set agar sebanding dengan R1/R2. Catat jenis runtime (CPU/GPU/TPU) di paper.

In [ ]:
X_train_final = X_train + X_val
y_train_final = np.concatenate([y_train, y_val])

X_train_final_R3 = np.vstack([X_train_R3, X_val_R3])

def final_representation_data(rep):
    if rep in ("R1", "R2"):
        return X_train_final, X_test
    return X_train_final_R3, X_test_R3

def binary_security_metrics(y_true, y_pred):
    true_attack = np.asarray(y_true) != 0
    pred_attack = np.asarray(y_pred) != 0
    tn = int(np.sum(~true_attack & ~pred_attack))
    fp = int(np.sum(~true_attack & pred_attack))
    fn = int(np.sum(true_attack & ~pred_attack))
    tp = int(np.sum(true_attack & pred_attack))
    return {
        "FPR_benign": fp / (fp + tn) if (fp + tn) else np.nan,
        "FNR_attack": fn / (fn + tp) if (fn + tp) else np.nan,
        "FP_benign": fp, "FN_attack": fn, "TN_benign": tn, "TP_attack": tp
    }

# Pemanasan model agar inisialisasi pertama tidak ikut terhitung.
_ = embedding_model.encode(X_test[:64], batch_size=64, convert_to_numpy=True,
                           show_progress_bar=False)
# Waktu encoding embedding test set (tanpa cache) agar biaya fitur R3 ikut terhitung.
start = time.perf_counter()
_ = embedding_model.encode(X_test, batch_size=64, convert_to_numpy=True,
                           show_progress_bar=False)
EMBED_TEST_SECONDS = time.perf_counter() - start
print(f"Encoding embedding test set: {EMBED_TEST_SECONDS:.2f} s")

final_models, final_predictions, final_scores = {}, {}, {}
final_rows = []

for (algorithm, representation), config in best_configs.items():
    seeds = SEEDS_FINAL if algorithm in ("Random Forest", "MLP") else [SEED]
    X_fit, X_eval = final_representation_data(representation)
    for seed in seeds:
        model = make_model(algorithm, representation, config["params"], seed, True)
        start = time.perf_counter()
        model.fit(X_fit, y_train_final)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        pred = model.predict(X_eval)
        inference_seconds = time.perf_counter() - start
        key = (algorithm, representation, seed)
        final_models[key] = model
        final_predictions[key] = pred
        try:
            scores = model.predict_proba(X_eval) if hasattr(model, "predict_proba") else model.decision_function(X_eval)
        except Exception:
            scores = None
        final_scores[key] = scores
        sec = binary_security_metrics(y_test, pred)
        final_rows.append({
            "algorithm": algorithm, "representation": representation, "seed": seed,
            "accuracy": accuracy_score(y_test, pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_test, pred, average="weighted", zero_division=0),
            "mcc": matthews_corrcoef(y_test, pred),
            "fit_seconds": fit_seconds,
            "inference_ms_per_1000": inference_seconds / len(y_test) * 1_000_000,
            "inference_ms_per_1000_incl_features": (
                inference_seconds + (EMBED_TEST_SECONDS if representation == "R3" else 0.0)
            ) / len(y_test) * 1_000_000,
            **sec
        })

final_results_df = pd.DataFrame(final_rows)
metric_cols = [
    "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mcc",
    "FPR_benign", "FNR_attack", "fit_seconds", "inference_ms_per_1000",
    "inference_ms_per_1000_incl_features"
]
final_summary = final_results_df.groupby(
    ["algorithm", "representation"]
)[metric_cols].agg(["mean", "std"]).round(4)
display(final_results_df)
display(final_summary)

## 08. Evaluasi per kelas dan confusion matrix

Metrik utama adalah Macro-F1 6 kelas. Analisis per kelas, per sumber, per kategori, dan McNemar memakai seed 42 untuk RF/MLP (tabel utama memakai rata-rata lima seed); sebutkan ini di paper. Metrik per kelas, confusion matrix ternormalisasi, dan FPR/FNR merupakan metrik pendukung yang diturunkan dari prediksi multikelas.

In [ ]:
representative_predictions = {}
for algorithm in ["Naive Bayes", "SVM", "Random Forest", "MLP"]:
    for representation in REPRESENTATIONS:
        representative_predictions[(algorithm, representation)] = final_predictions[
            (algorithm, representation, SEED)
        ]

per_class_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    report = classification_report(
        y_test, pred, labels=CLASS_IDS, output_dict=True, zero_division=0
    )
    for class_id in CLASS_IDS:
        row = report[str(class_id)]
        per_class_rows.append({
            "algorithm": algorithm, "representation": representation,
            "class_code": f"K{class_id}",
            "class_name": class_names[f"K{class_id}"],
            "precision": row["precision"], "recall": row["recall"],
            "f1": row["f1-score"], "support": row["support"]
        })
per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)

validation_rank = best_config_df.sort_values("validation_macro_f1", ascending=False)
top_valid = validation_rank.iloc[0]
top_key = (top_valid["algorithm"], top_valid["representation"], SEED)
top_pred = final_predictions[top_key]
cm = confusion_matrix(y_test, top_pred, labels=CLASS_IDS, normalize="true")
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=[f"K{i}" for i in CLASS_IDS],
    yticklabels=[f"K{i}" for i in CLASS_IDS]
)
plt.title(f"Normalized Confusion Matrix — {top_valid['algorithm']} / {top_valid['representation']}")
plt.xlabel("Predicted class"); plt.ylabel("True class"); plt.tight_layout()
save_figure("confusion_matrix_best_validation_model.png")
plt.show(); plt.close()

## 09. A3 — Analisis per sumber dan subset sulit

Macro-F1 enam kelas **tidak** digunakan per sumber karena banyak sumber hanya berisi satu atau dua kelas; label yang tidak hadir akan menghasilkan skor yang menyesatkan. Sebagai gantinya, laporkan accuracy per sumber dan FPR/FNR bila sumber memiliki kelas benign/serangan yang relevan.

**Hard negative** = prompt benign (K0) di test, dari sumber mana pun, yang memuat kata pemicu yang lazim pada serangan (`ignore`, `disregard`, `forget`, `override`, `bypass`, `jailbreak`, `pretend`, `instruction(s)`, `system prompt`). FPR pada subset ini mengukur salah blokir terhadap prompt aman yang mirip serangan secara leksikal. Laporkan jumlah sampelnya; bila kecil (< 20), tulis sebagai indikasi, bukan kesimpulan.

In [ ]:
source_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    source_values = test["source"].fillna("unknown").astype(str).to_numpy()
    for source in sorted(set(source_values)):
        mask = source_values == source
        if not mask.any():
            continue
        y_sub, p_sub = y_test[mask], pred[mask]
        benign_mask = y_sub == 0
        attack_mask = y_sub != 0
        fpr = float(np.mean(p_sub[benign_mask] != 0)) if benign_mask.any() else np.nan
        fnr = float(np.mean(p_sub[attack_mask] == 0)) if attack_mask.any() else np.nan
        source_rows.append({
            "algorithm": algorithm, "representation": representation,
            "source": source, "support": int(mask.sum()),
            "accuracy_6class": accuracy_score(y_sub, p_sub),
            "FPR_benign": fpr, "FNR_attack": fnr,
            "benign_support": int(benign_mask.sum()),
            "attack_support": int(attack_mask.sum())
        })
source_results_df = pd.DataFrame(source_rows)
display(source_results_df)

TRIGGER_PATTERN = re.compile(
    r"\b(?:ignore|disregard|forget|override|bypass|jailbreak|pretend|instructions?|system prompt)\b",
    flags=re.I
)
hard_negative_mask = (
    test["class_id"].eq(0).to_numpy()
    & test["text_clean"].astype(str).str.contains(TRIGGER_PATTERN).to_numpy()
)
print("Hard negative (benign + kata pemicu) di test:", int(hard_negative_mask.sum()))
display(test.loc[hard_negative_mask, "source"].value_counts().rename("hard_negative_rows"))
if hard_negative_mask.sum() < 20:
    print("PERINGATAN: subset hard negative kecil; laporkan sebagai indikasi.")
hard_negative_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    n = int(hard_negative_mask.sum())
    fp = int(np.sum(pred[hard_negative_mask] != 0)) if n else 0
    hard_negative_rows.append({
        "algorithm": algorithm, "representation": representation,
        "hard_negative_support": n,
        "false_block_count": fp,
        "FPR_hard_negative": fp / n if n else np.nan
    })
hard_negative_df = pd.DataFrame(hard_negative_rows)
display(hard_negative_df)

## 10. A10 — Recall per kategori asli (heatmap)

Recall dihitung sebagai persentase baris kategori asli yang diprediksi ke kelas hasil mapping yang benar. Ini berbeda dari deteksi biner serangan (prediksi bukan K0). Heatmap menyusun kolom per algoritma lalu representasi (R1/R2/R3 bersebelahan) sehingga langsung menjawab RQ2: representasi mana yang membantu tiap teknik. Kategori dengan support kecil (n tertera di label) harus dibaca hati-hati.

In [ ]:
category_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    categories = test["category"].astype(str).to_numpy()
    for category in sorted(set(categories)):
        mask = categories == category
        true_ids, pred_ids = y_test[mask], pred[mask]
        category_rows.append({
            "algorithm": algorithm, "representation": representation,
            "original_category": category, "support": int(mask.sum()),
            "mapped_class": f"K{int(pd.Series(true_ids).mode().iloc[0])}",
            "category_recall": float(np.mean(pred_ids == true_ids))
        })
category_level_df = pd.DataFrame(category_rows)
display(category_level_df.sort_values(["original_category", "algorithm", "representation"]))

ALGORITHM_ORDER = ["Naive Bayes", "SVM", "Random Forest", "MLP"]
recall_plot_df = category_level_df.copy()
recall_plot_df["row_label"] = (
    recall_plot_df["mapped_class"] + " · " + recall_plot_df["original_category"]
    + " (n=" + recall_plot_df["support"].astype(str) + ")"
)
recall_pivot = recall_plot_df.pivot_table(
    index="row_label", columns=["algorithm", "representation"], values="category_recall"
).reindex(columns=pd.MultiIndex.from_product([ALGORITHM_ORDER, REPRESENTATIONS]))
recall_pivot = recall_pivot.sort_index()
recall_pivot.columns = [f"{a} {r}" for a, r in recall_pivot.columns]
plt.figure(figsize=(15, max(7, 0.38 * len(recall_pivot))))
sns.heatmap(recall_pivot, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="YlGnBu",
            cbar_kws={"label": "Recall"})
plt.title("Recall by Original Category (test, seed 42)")
plt.xlabel("Algorithm / Representation"); plt.ylabel("Mapped class · original category")
plt.tight_layout()
save_figure("recall_by_original_category_heatmap.png")
plt.show(); plt.close()

# Rincian K5: apakah kinerja K5 didorong oleh payload kode (adversarial)
# atau juga berlaku untuk injeksi tidak langsung / agen?
k5_breakdown_df = category_level_df[category_level_df["mapped_class"] == "K5"].pivot_table(
    index=["original_category", "support"], columns=["algorithm", "representation"],
    values="category_recall"
)
display(k5_breakdown_df.round(3))

## 11. A2 — Eksperimen biner K0 vs K1–K5

Versi biner dibangun dari label enam kelas: K0 = benign, K1–K5 = attack. Hasil dibandingkan secara deskriptif dengan baseline dataset card: Random Forest F1 0,969 / FPR 6,9%; LinearSVC F1 0,959 / FPR 10,3%. Perbandingan harus menyebutkan perbedaan protokol dan subset data.

In [ ]:
y_train_binary = (y_train != 0).astype(int)
y_val_binary = (y_val != 0).astype(int)
y_test_binary = (y_test != 0).astype(int)

binary_rows = []
binary_models = {}
for (algorithm, representation), config in best_configs.items():
    # Pilih classifier family yang sama, tetapi tune langsung pada label biner.
    Xtr, Xva, Xte = representation_data(representation)
    Xfit = list(Xtr) + list(Xva) if representation in ("R1", "R2") else np.vstack([Xtr, Xva])
    yfit = np.concatenate([y_train_binary, y_val_binary])
    best_score, best_params = -np.inf, None
    for params in ParameterGrid(PARAM_GRIDS[algorithm][representation]):
        clf = make_model(algorithm, representation, params, SEED, True)
        split = PredefinedSplit(np.r_[np.full(len(y_train_binary), -1), np.zeros(len(y_val_binary))])
        tr_idx, va_idx = next(split.split())
        Xtr_fold = [Xfit[i] for i in tr_idx] if isinstance(Xfit, list) else Xfit[tr_idx]
        Xva_fold = [Xfit[i] for i in va_idx] if isinstance(Xfit, list) else Xfit[va_idx]
        clf.fit(Xtr_fold, yfit[tr_idx])
        score = f1_score(y_val_binary, clf.predict(Xva_fold), average="binary", zero_division=0)
        if score > best_score:
            best_score, best_params = score, params.copy()
    X_final, X_eval = final_representation_data(representation)
    final_model = make_model(algorithm, representation, best_params, SEED, True)
    final_model.fit(X_final, np.concatenate([y_train_binary, y_val_binary]))
    pred = final_model.predict(X_eval)
    binary_auc = np.nan
    if hasattr(final_model, "predict_proba"):
        probabilities = final_model.predict_proba(X_eval)
        positive_col = list(final_model.named_steps["classifier"].classes_).index(1)
        binary_auc = roc_auc_score(y_test_binary, probabilities[:, positive_col])
    elif hasattr(final_model, "decision_function"):
        # LinearSVC/SVC tanpa probability: skor jarak ke hyperplane cukup untuk AUC.
        binary_auc = roc_auc_score(y_test_binary, final_model.decision_function(X_eval))
    binary_scores = None
    if hasattr(final_model, "predict_proba"):
        binary_scores = probabilities[:, positive_col]
    elif hasattr(final_model, "decision_function"):
        binary_scores = final_model.decision_function(X_eval)
    binary_models[(algorithm, representation)] = {"model": final_model, "scores": binary_scores}
    sec = binary_security_metrics(y_test, np.where(pred == 1, 1, 0))
    binary_rows.append({
        "algorithm": algorithm, "representation": representation,
        "validation_binary_f1": best_score,
        "test_binary_f1": f1_score(y_test_binary, pred, zero_division=0),
        "test_accuracy": accuracy_score(y_test_binary, pred),
        "roc_auc_attack": binary_auc,
        "FPR_benign": sec["FPR_benign"], "FNR_attack": sec["FNR_attack"],
        "params": str(best_params)
    })
binary_results_df = pd.DataFrame(binary_rows)
display(binary_results_df)

dataset_card_baselines = pd.DataFrame([
    {"model": "Random Forest (dataset-card baseline)", "F1": 0.969, "FPR": 0.069},
    {"model": "Logistic Regression (dataset-card baseline)", "F1": 0.964, "FPR": 0.064},
    {"model": "LinearSVC (dataset-card baseline)", "F1": 0.959, "FPR": 0.103}
])
display(dataset_card_baselines)

## 12. A3 — F1 K2 dengan dan tanpa baris HarmBench

Dataset mencatat 70 baris HarmBench yang berlabel jailbreak tetapi tidak memuat teknik jailbreak yang jelas. Analisis ini mengukur sensitivitas F1 K2 terhadap subset tersebut. Nilai harus dibaca sebagai audit label/sumber, bukan sebagai hasil model terpisah.

In [ ]:
harmbench_mask = test["source"].astype(str).str.lower().str.contains("harmbench", na=False).to_numpy()
k2_harmbench_mask = (y_test == 2) & harmbench_mask
keep_without_k2_harmbench = ~k2_harmbench_mask

k2_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    f1_all = f1_score(
        (y_test == 2).astype(int), (pred == 2).astype(int),
        average="binary", zero_division=0
    )
    f1_without = f1_score(
        (y_test[keep_without_k2_harmbench] == 2).astype(int),
        (pred[keep_without_k2_harmbench] == 2).astype(int),
        average="binary", zero_division=0
    )
    k2_rows.append({
        "algorithm": algorithm, "representation": representation,
        "K2_support_total": int(np.sum(y_test == 2)),
        "K2_harmbench_support": int(k2_harmbench_mask.sum()),
        "K2_f1_with_harmbench": f1_all,
        "K2_f1_without_harmbench": f1_without
    })
k2_harmbench_df = pd.DataFrame(k2_rows)
display(k2_harmbench_df)

### 12.1 A3 — Ketahanan terhadap augmentasi

Dataset memuat baris augmentasi (homoglyph, leetspeak, sisipan tanda baca) yang diterapkan ke banyak kategori, bukan hanya K3. Karena itu metrik dilaporkan juga pada subset test **non-augmentasi** dan **augmentasi**. Macro-F1 dihitung hanya atas kelas yang hadir di subset tersebut (jumlah kelas dicatat).

In [ ]:
augmented_flag = test["augmented"]
if augmented_flag.dtype != bool:
    augmented_flag = augmented_flag.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
augmented_flag = augmented_flag.to_numpy()

augmentation_share_df = pd.crosstab(
    pd.concat([train, val, test], ignore_index=True)["class_code"],
    pd.concat([train, val, test], ignore_index=True)["augmented"].astype(str),
    margins=True
)
display(augmentation_share_df)
print("Test non-augmentasi:", int((~augmented_flag).sum()), "dari", len(test))

augmentation_rows = []
subsets = {
    "all": np.ones(len(test), dtype=bool),
    "non_augmented": ~augmented_flag,
    "augmented": augmented_flag,
}
for (algorithm, representation), pred in representative_predictions.items():
    for subset_name, mask in subsets.items():
        if not mask.any():
            continue
        y_sub, p_sub = y_test[mask], pred[mask]
        present = np.unique(y_sub)
        augmentation_rows.append({
            "algorithm": algorithm, "representation": representation,
            "subset": subset_name, "support": int(mask.sum()),
            "n_classes_present": len(present),
            "macro_f1_present_classes": f1_score(
                y_sub, p_sub, labels=present, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_sub, p_sub),
            **{k: v for k, v in binary_security_metrics(y_sub, p_sub).items()
               if k in ("FPR_benign", "FNR_attack")}
        })
augmentation_robustness_df = pd.DataFrame(augmentation_rows)
display(augmentation_robustness_df.pivot_table(
    index=["algorithm", "representation"], columns="subset",
    values="macro_f1_present_classes"
).round(4))

## 13. A1 — Ablation `class_weight`

Eksperimen ablation membandingkan konfigurasi SVM dan Random Forest dengan `class_weight="balanced"` versus tanpa class weight. Hyperparameter utama yang terpilih pada konfigurasi balanced dipertahankan agar yang diuji hanya efek class weight.

In [ ]:
ablation_rows = []
for algorithm in ["SVM", "Random Forest"]:
    for representation in REPRESENTATIONS:
        params = best_configs[(algorithm, representation)]["params"]
        X_fit, X_eval = final_representation_data(representation)
        for use_weight in [True, False]:
            model = make_model(algorithm, representation, params, SEED, use_weight)
            model.fit(X_fit, y_train_final)
            pred = model.predict(X_eval)
            sec = binary_security_metrics(y_test, pred)
            ablation_rows.append({
                "algorithm": algorithm, "representation": representation,
                "class_weight": "balanced" if use_weight else "none",
                "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
                "accuracy": accuracy_score(y_test, pred),
                **sec
            })
class_weight_ablation_df = pd.DataFrame(ablation_rows)
display(class_weight_ablation_df)

## 14. Uji statistik dan ketidakpastian

- Bootstrap terstratifikasi per kelas 1.000 kali untuk CI 95% Macro-F1 **ke-12 konfigurasi** (seed 42). Resampel yang sama dipakai untuk semua model (berpasangan). CI F1 per kelas dilaporkan untuk model terbaik; perhatikan K4 yang support test-nya kecil.
- McNemar exact untuk model terbaik vs pesaing berdasarkan validation.
- McNemar antar-representasi R1–R2, R1–R3, R2–R3 pada setiap algoritma; koreksi Holm diterapkan pada keluarga pengujian.
- McNemar menguji perbedaan correctness berpasangan, bukan perbedaan Macro-F1 secara langsung.
- Keluarga Holm = 12 uji antar-representasi (4 algoritma × 3 pasangan). Perbedaan yang tidak signifikan setelah koreksi tidak boleh ditulis sebagai keunggulan.

In [ ]:
# Bootstrap CI berpasangan untuk ke-12 konfigurasi (seed 42).
N_BOOT = 1000
rng = np.random.default_rng(SEED)
class_indices = [np.where(y_test == cid)[0] for cid in CLASS_IDS]
boot_samples = [
    np.concatenate([rng.choice(idx, size=len(idx), replace=True)
                    for idx in class_indices if len(idx) > 0])
    for _ in range(N_BOOT)
]

bootstrap_all_rows = []
for (algorithm, representation), pred in representative_predictions.items():
    scores = [f1_score(y_test[s], pred[s], labels=CLASS_IDS, average="macro", zero_division=0)
              for s in boot_samples]
    bootstrap_all_rows.append({
        "algorithm": algorithm, "representation": representation,
        "observed_macro_f1": f1_score(y_test, pred, labels=CLASS_IDS, average="macro", zero_division=0),
        "ci_2_5": np.percentile(scores, 2.5), "ci_97_5": np.percentile(scores, 97.5),
        "bootstrap_replicates": N_BOOT
    })
bootstrap_all_df = pd.DataFrame(bootstrap_all_rows).sort_values("observed_macro_f1", ascending=False)
display(bootstrap_all_df.round(4))

top_row = bootstrap_all_df[
    (bootstrap_all_df["algorithm"] == top_valid["algorithm"])
    & (bootstrap_all_df["representation"] == top_valid["representation"])
].iloc[0]
bootstrap_ci = pd.DataFrame([{
    "configuration": f"{top_valid['algorithm']} / {top_valid['representation']}",
    "observed_macro_f1": top_row["observed_macro_f1"],
    "ci_2_5": top_row["ci_2_5"], "ci_97_5": top_row["ci_97_5"],
    "bootstrap_replicates": N_BOOT
}])
display(bootstrap_ci)

# CI F1 per kelas untuk model terbaik menurut validation.
per_class_boot = np.array([
    f1_score(y_test[s], top_pred[s], labels=CLASS_IDS, average=None, zero_division=0)
    for s in boot_samples
])
top_per_class_ci_df = pd.DataFrame({
    "class_code": CLASS_CODES,
    "class_name": [class_names[c] for c in CLASS_CODES],
    "support": [int(np.sum(y_test == cid)) for cid in CLASS_IDS],
    "f1": f1_score(y_test, top_pred, labels=CLASS_IDS, average=None, zero_division=0),
    "ci_2_5": np.percentile(per_class_boot, 2.5, axis=0),
    "ci_97_5": np.percentile(per_class_boot, 97.5, axis=0),
})
display(top_per_class_ci_df.round(4))

def exact_mcnemar(pred_a, pred_b, y_true):
    correct_a, correct_b = pred_a == y_true, pred_b == y_true
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    p = binomtest(min(b, c), n=b+c, p=0.5, alternative="two-sided").pvalue if b+c else 1.0
    return b, c, p

# Model terbaik dan pesaing berdasarkan validation.
top_two = validation_rank.head(2)
best_competitor_rows = []
if len(top_two) == 2:
    a, b = top_two.iloc[0], top_two.iloc[1]
    pa = final_predictions[(a["algorithm"], a["representation"], SEED)]
    pb = final_predictions[(b["algorithm"], b["representation"], SEED)]
    discord_a, discord_b, p = exact_mcnemar(pa, pb, y_test)
    best_competitor_rows.append({
        "comparison": "best_vs_runner_up",
        "model_a": f"{a['algorithm']} / {a['representation']}",
        "model_b": f"{b['algorithm']} / {b['representation']}",
        "a_correct_b_wrong": discord_a, "a_wrong_b_correct": discord_b, "p_raw": p
    })
best_competitor_mcnemar_df = pd.DataFrame(best_competitor_rows)
display(best_competitor_mcnemar_df)

representation_tests = []
for algorithm in ["Naive Bayes", "SVM", "Random Forest", "MLP"]:
    for rep_a, rep_b in [("R1","R2"), ("R1","R3"), ("R2","R3")]:
        pred_a = final_predictions[(algorithm, rep_a, SEED)]
        pred_b = final_predictions[(algorithm, rep_b, SEED)]
        b, c, p = exact_mcnemar(pred_a, pred_b, y_test)
        representation_tests.append({
            "algorithm": algorithm, "representation_a": rep_a,
            "representation_b": rep_b, "a_correct_b_wrong": b,
            "a_wrong_b_correct": c, "p_raw": p
        })
representation_mcnemar_df = pd.DataFrame(representation_tests)
representation_mcnemar_df["p_holm"] = multipletests(
    representation_mcnemar_df["p_raw"], method="holm"
)[1]
display(representation_mcnemar_df)

## 15. A7 — Interpretabilitas fitur

Fitur teratas dilaporkan hanya untuk R1/R2:
- LinearSVC: koefisien per kelas.
- ComplementNB: bobot `feature_log_prob_` dikurangi rata-ratanya di semua kelas (bobot terpusat). Bobot mentah ComplementNB diturunkan dari kelas komplemen dan cenderung tinggi untuk token yang jarang di mana pun, sehingga tidak dapat dibaca langsung sebagai "log-probabilitas per kelas".
- Random Forest: feature importance global.
- R3 tidak memiliki fitur leksikal yang dapat ditafsirkan dengan cara yang sama. MLP tidak diberi klaim feature importance langsung.

In [ ]:
feature_rows = []
for representation in ["R1", "R2"]:
    # LinearSVC
    svm_model = final_models[("SVM", representation, SEED)]
    vectorizer = svm_model.named_steps["tfidf"]
    feature_names = vectorizer.get_feature_names_out()
    svm_clf = svm_model.named_steps["classifier"]
    for class_pos, class_id in enumerate(svm_clf.classes_):
        coef = svm_clf.coef_[class_pos]
        for rank, idx in enumerate(np.argsort(coef)[-20:][::-1], start=1):
            feature_rows.append({
                "algorithm": "LinearSVC", "representation": representation,
                "class_code": f"K{class_id}", "rank": rank,
                "feature": feature_names[idx], "importance": float(coef[idx])
            })

    # ComplementNB: prediksi = argmax_c sum_w x_w * W[c, w]; yang membedakan kelas
    # adalah selisih bobot terhadap rata-rata antar-kelas (bobot terpusat).
    nb_model = final_models[("Naive Bayes", representation, SEED)]
    nb_clf = nb_model.named_steps["classifier"]
    nb_weights = nb_clf.feature_log_prob_
    nb_centered = nb_weights - nb_weights.mean(axis=0, keepdims=True)
    for class_pos, class_id in enumerate(nb_clf.classes_):
        values = nb_centered[class_pos]
        for rank, idx in enumerate(np.argsort(values)[-20:][::-1], start=1):
            feature_rows.append({
                "algorithm": "ComplementNB (centered)", "representation": representation,
                "class_code": f"K{class_id}", "rank": rank,
                "feature": feature_names[idx], "importance": float(values[idx])
            })

    # Random Forest: global importance (bukan per kelas).
    rf_model = final_models[("Random Forest", representation, SEED)]
    rf_clf = rf_model.named_steps["classifier"]
    for rank, idx in enumerate(np.argsort(rf_clf.feature_importances_)[-30:][::-1], start=1):
        feature_rows.append({
            "algorithm": "RandomForest", "representation": representation,
            "class_code": "GLOBAL", "rank": rank,
            "feature": feature_names[idx],
            "importance": float(rf_clf.feature_importances_[idx])
        })

top_features_df = pd.DataFrame(feature_rows)
display(top_features_df)

## 15A. Analisis tambahan untuk paper (revisi 8 Nov)

Sel-sel berikut menghasilkan semua angka di paper yang sebelumnya **belum** punya CSV: bootstrap berpasangan selisih Macro-F1, McNemar SVM vs RF, uji per *seed*, recall per kelas per sumber (dominasi hackaprompt pada K1), derau label wildguard, uji sumber yang disisihkan (*held-out source*), NotInject, TPR pada FPR rendah (varian biner), dan cek *near-duplicate* antar-split. Setiap sel menyimpan CSV ke `results/tables/`.


In [ ]:
# 15A.1 Bootstrap berpasangan selisih Macro-F1 + Holm per keluarga uji.
# Memakai boot_samples yang sama dengan CI di bagian 14 (berpasangan).
def _macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=CLASS_IDS, average="macro", zero_division=0)

def paired_bootstrap_diff(pred_a, pred_b):
    observed = _macro_f1(y_test, pred_a) - _macro_f1(y_test, pred_b)
    diffs = np.array([_macro_f1(y_test[s], pred_a[s]) - _macro_f1(y_test[s], pred_b[s])
                      for s in boot_samples])
    n_le, n_ge = int(np.sum(diffs <= 0)), int(np.sum(diffs >= 0))
    p = min(1.0, 2 * (min(n_le, n_ge) + 1) / (len(diffs) + 1))
    return observed, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), p

paired_rows = []
# Keluarga 1: 12 pasangan representasi (seed 42), sama dengan keluarga McNemar.
for algorithm in ALGORITHM_ORDER:
    for rep_a, rep_b in [("R2", "R1"), ("R2", "R3"), ("R1", "R3")]:
        obs, lo, hi, p = paired_bootstrap_diff(
            final_predictions[(algorithm, rep_a, SEED)],
            final_predictions[(algorithm, rep_b, SEED)])
        paired_rows.append({"family": "representation", "model_a": f"{algorithm} {rep_a}",
                            "model_b": f"{algorithm} {rep_b}", "macro_f1_diff": obs,
                            "ci_2_5": lo, "ci_97_5": hi, "p_raw": p})
# Keluarga 2: model terbaik vs dua pesaing pada R2 (seed 42).
for rival in ["Random Forest", "MLP"]:
    obs, lo, hi, p = paired_bootstrap_diff(
        final_predictions[("SVM", "R2", SEED)], final_predictions[(rival, "R2", SEED)])
    paired_rows.append({"family": "best_vs_rivals", "model_a": "SVM R2",
                        "model_b": f"{rival} R2", "macro_f1_diff": obs,
                        "ci_2_5": lo, "ci_97_5": hi, "p_raw": p})
paired_bootstrap_df = pd.DataFrame(paired_rows)
paired_bootstrap_df["p_holm"] = np.nan
for fam, idx in paired_bootstrap_df.groupby("family").groups.items():
    paired_bootstrap_df.loc[idx, "p_holm"] = multipletests(
        paired_bootstrap_df.loc[idx, "p_raw"], method="holm")[1]
display(paired_bootstrap_df.round(4))
save_table(paired_bootstrap_df, "bootstrap_paired_macro_f1_diff.csv")


In [ ]:
# 15A.2 McNemar SVM R2 vs RF R2 dan vs MLP R2 (seed 42).
rival_mcnemar_rows = []
for rival in ["Random Forest", "MLP"]:
    b, c, p = exact_mcnemar(final_predictions[("SVM", "R2", SEED)],
                            final_predictions[(rival, "R2", SEED)], y_test)
    rival_mcnemar_rows.append({"model_a": "SVM R2", "model_b": f"{rival} R2",
                               "a_correct_b_wrong": b, "a_wrong_b_correct": c, "p_raw": p})
rival_mcnemar_df = pd.DataFrame(rival_mcnemar_rows)
display(rival_mcnemar_df)
save_table(rival_mcnemar_df, "mcnemar_svm_vs_rivals_r2.csv")


In [ ]:
# 15A.3 Uji per seed (42-46) untuk RF dan MLP: apakah kesimpulan bergantung pada seed?
per_seed_rows = []
for seed in SEEDS_FINAL:
    comparisons = [
        ("SVM R2 vs RF R2", final_predictions[("SVM", "R2", SEED)],
         final_predictions[("Random Forest", "R2", seed)]),
        ("SVM R2 vs MLP R2", final_predictions[("SVM", "R2", SEED)],
         final_predictions[("MLP", "R2", seed)]),
        ("RF R2 vs RF R1", final_predictions[("Random Forest", "R2", seed)],
         final_predictions[("Random Forest", "R1", seed)]),
        ("MLP R2 vs MLP R1", final_predictions[("MLP", "R2", seed)],
         final_predictions[("MLP", "R1", seed)]),
        ("RF R2 vs RF R3", final_predictions[("Random Forest", "R2", seed)],
         final_predictions[("Random Forest", "R3", seed)]),
        ("MLP R2 vs MLP R3", final_predictions[("MLP", "R2", seed)],
         final_predictions[("MLP", "R3", seed)]),
    ]
    for name, pa, pb in comparisons:
        obs, lo, hi, p_boot = paired_bootstrap_diff(pa, pb)
        b, c, p_mc = exact_mcnemar(pa, pb, y_test)
        per_seed_rows.append({"seed": seed, "comparison": name, "macro_f1_diff": obs,
                              "ci_2_5": lo, "ci_97_5": hi, "p_bootstrap_raw": p_boot,
                              "ci_excludes_zero": bool(lo > 0 or hi < 0),
                              "mcnemar_b": b, "mcnemar_c": c, "p_mcnemar_raw": p_mc})
per_seed_tests_df = pd.DataFrame(per_seed_rows)
display(per_seed_tests_df.pivot_table(index="comparison", columns="seed",
                                      values="ci_excludes_zero", aggfunc="first"))
save_table(per_seed_tests_df, "per_seed_significance_tests.csv")


In [ ]:
# 15A.4 Recall per kelas per sumber untuk model terbaik (SVM R2).
# Menjawab: apakah F1 K1 yang tinggi didorong oleh satu sumber (hackaprompt)?
best_pred_svm_r2 = final_predictions[("SVM", "R2", SEED)]
test_sources = test["source"].fillna("unknown").astype(str).to_numpy()
class_source_rows = []
for cid in CLASS_IDS:
    for source in sorted(set(test_sources)):
        mask = (y_test == cid) & (test_sources == source)
        if mask.any():
            class_source_rows.append({
                "class_code": f"K{cid}", "source": source, "support": int(mask.sum()),
                "correct": int(np.sum(best_pred_svm_r2[mask] == cid)),
                "recall": float(np.mean(best_pred_svm_r2[mask] == cid))})
class_source_recall_df = pd.DataFrame(class_source_rows)
display(class_source_recall_df)
k1 = class_source_recall_df[class_source_recall_df["class_code"] == "K1"]
k1_hack = k1[k1["source"] == "hackaprompt"][["correct", "support"]].sum()
k1_rest = k1[k1["source"] != "hackaprompt"][["correct", "support"]].sum()
print(f"Recall K1 hackaprompt: {k1_hack['correct']}/{k1_hack['support']}; "
      f"sumber lain: {k1_rest['correct']}/{k1_rest['support']}")
save_table(class_source_recall_df, "per_class_recall_by_source_best_model.csv")


In [ ]:
# 15A.5 Derau label wildguard: FPR sumber minoritas dengan dan tanpa baris yang dinilai keliru.
# Input: inputs/label_audit_wildguard.csv (penilaian tim per baris; kolom awal_teks, penilaian_tim).
DOMINANT_SOURCES = {"neuralchemy_v1", "hackaprompt"}
MINORITY_SOURCES = sorted(set(test_sources) - DOMINANT_SOURCES)
print("Sumber minoritas:", MINORITY_SOURCES)

label_audit_path = INPUTS_DIR / "label_audit_wildguard.csv"
benign_minority = (y_test == 0) & np.isin(test_sources, MINORITY_SOURCES)
noisy_mask = np.zeros(len(test), dtype=bool)
if label_audit_path.exists():
    label_audit_df = pd.read_csv(label_audit_path)
    keliru = label_audit_df[label_audit_df["penilaian_tim"].str.strip().str.lower() == "label keliru"]
    texts = test["text_clean"].astype(str).to_numpy()
    for prefix in keliru["awal_teks"].astype(str).map(preprocess_text):
        hits = np.where(benign_minority & (test_sources == "wildguard_judgecomp")
                        & np.array([t.startswith(prefix) for t in texts]))[0]
        if len(hits) != 1:
            print(f"PERINGATAN: prefix cocok {len(hits)} baris: {prefix[:60]}")
        noisy_mask[hits] = True
    print("Baris berlabel keliru yang cocok di test:", int(noisy_mask.sum()), "dari", len(keliru))
else:
    print(f"PERINGATAN: {label_audit_path} tidak ada; analisis derau label dilewati.")

label_noise_rows = []
for name, mask in [("label_asli", benign_minority),
                   ("tanpa_label_keliru", benign_minority & ~noisy_mask)]:
    fp = int(np.sum(best_pred_svm_r2[mask] != 0))
    label_noise_rows.append({"subset": name, "benign_minority_support": int(mask.sum()),
                             "false_positive": fp, "FPR": fp / mask.sum() if mask.any() else np.nan})
label_noise_df = pd.DataFrame(label_noise_rows)
display(label_noise_df)
save_table(label_noise_df, "label_noise_wildguard_fpr.csv")


In [ ]:
# 15A.6 Uji sumber yang disisihkan (held-out source) untuk model terbaik (SVM R2).
# Setiap sumber minoritas dikeluarkan dari train+validation, model dilatih ulang dengan
# hiperparameter yang sama, lalu diuji pada baris test dari sumber itu.
# Varian "harmbench+harmbench_benign" menahan dua sumber bertopik sama sekaligus.
train_final_sources = np.concatenate([
    train["source"].fillna("unknown").astype(str).to_numpy(),
    val["source"].fillna("unknown").astype(str).to_numpy()])
assert len(train_final_sources) == len(X_train_final)
svm_r2_params = best_configs[("SVM", "R2")]["params"]

holdout_groups = [[s] for s in MINORITY_SOURCES]
if {"harmbench", "harmbench_benign"}.issubset(MINORITY_SOURCES):
    holdout_groups.append(["harmbench", "harmbench_benign"])

heldout_rows = []
for group in holdout_groups:
    keep = ~np.isin(train_final_sources, group)
    model = make_model("SVM", "R2", svm_r2_params, SEED, True)
    model.fit([X_train_final[i] for i in np.where(keep)[0]], y_train_final[keep])
    for source in group:
        mask = test_sources == source
        pred_out = model.predict([X_test[i] for i in np.where(mask)[0]])
        pred_in = best_pred_svm_r2[mask]
        y_src = y_test[mask]
        benign = y_src == 0
        heldout_rows.append({
            "held_out": "+".join(group), "eval_source": source,
            "support": int(mask.sum()), "benign_support": int(benign.sum()),
            "accuracy_in_distribution": accuracy_score(y_src, pred_in),
            "accuracy_held_out": accuracy_score(y_src, pred_out),
            "fp_in_distribution": int(np.sum(pred_in[benign] != 0)),
            "fp_held_out": int(np.sum(pred_out[benign] != 0)),
            "excluded_train_rows": int((~keep).sum())})
heldout_df = pd.DataFrame(heldout_rows)
display(heldout_df)
single = heldout_df[~heldout_df["held_out"].str.contains(r"\+")]
print("FP benign minoritas, in-distribution:", int(single["fp_in_distribution"].sum()),
      "/", int(single["benign_support"].sum()),
      "| held-out:", int(single["fp_held_out"].sum()), "/", int(single["benign_support"].sum()))
save_table(heldout_df, "heldout_source_results.csv")


In [ ]:
# 15A.7 NotInject (Li dkk., InjecGuard; arXiv 2410.22770): 339 prompt benign berkata pemicu.
# FPR = proporsi yang diprediksi sebagai kelas serangan apa pun (bukan K0).
notinject = load_dataset("leolee99/NotInject")
notinject_df = pd.concat(
    [notinject[split].to_pandas().assign(subset=split) for split in notinject.keys()],
    ignore_index=True)
X_notinject = notinject_df["prompt"].map(preprocess_text).astype(str).tolist()
print("NotInject rows:", len(X_notinject))
assert len(X_notinject) == 339, "Ukuran NotInject berbeda dari 339; periksa versi dataset."
X_notinject_R3 = embedding_model.encode(X_notinject, batch_size=64, convert_to_numpy=True,
                                        show_progress_bar=False).astype(np.float32)

notinject_rows = []
for algorithm in ALGORITHM_ORDER:
    for representation in REPRESENTATIONS:
        model = final_models[(algorithm, representation, SEED)]
        X_eval = X_notinject if representation in ("R1", "R2") else X_notinject_R3
        pred = model.predict(X_eval)
        notinject_rows.append({
            "algorithm": algorithm, "representation": representation,
            "n": len(pred), "flagged_as_attack": int(np.sum(pred != 0)),
            "FPR": float(np.mean(pred != 0)),
            **{f"pred_K{c}": int(np.sum(pred == c)) for c in CLASS_IDS}})
notinject_results_df = pd.DataFrame(notinject_rows)
display(notinject_results_df)
save_table(notinject_results_df, "notinject_overdefense_results.csv")


In [ ]:
# 15A.8 TPR pada FPR rendah untuk varian biner (sebanding dengan pelaporan PromptShield).
from sklearn.metrics import roc_curve
tpr_rows = []
for (algorithm, representation), info in binary_models.items():
    scores_bin = info["scores"]
    if scores_bin is None:
        continue
    fpr_curve, tpr_curve, _ = roc_curve(y_test_binary, scores_bin)
    row = {"algorithm": algorithm, "representation": representation}
    for target in [0.01, 0.05]:
        ok = fpr_curve <= target
        row[f"TPR_at_FPR_{int(target * 100)}pct"] = float(tpr_curve[ok].max()) if ok.any() else 0.0
    tpr_rows.append(row)
tpr_at_fpr_df = pd.DataFrame(tpr_rows)
display(tpr_at_fpr_df.round(4))
save_table(tpr_at_fpr_df, "binary_tpr_at_low_fpr.csv")


In [ ]:
# 15A.9 Cek near-duplicate antara test dan train+validation (TF-IDF karakter, kosinus).
dup_vectorizer = TfidfVectorizer(**TFIDF_CONFIG["R2"])
dup_train = dup_vectorizer.fit_transform(X_train_final)
dup_test = dup_vectorizer.transform(X_test)
max_sim = np.asarray((dup_test @ dup_train.T).max(axis=1).todense()).ravel()
train_text_set = set(X_train_final)
near_dup_df = pd.DataFrame([{
    "test_rows": len(X_test),
    "exact_duplicates": int(sum(t in train_text_set for t in X_test)),
    "cosine_ge_0_95": int(np.sum(max_sim >= 0.95)),
    "cosine_ge_0_90": int(np.sum(max_sim >= 0.90)),
    "median_max_cosine": float(np.median(max_sim))}])
display(near_dup_df)
near_dup_by_class = (pd.DataFrame({"class_code": [f"K{c}" for c in y_test],
                                   "near_dup_090": max_sim >= 0.90})
                     .groupby("class_code")["near_dup_090"].agg(["sum", "count"]))
display(near_dup_by_class)
save_table(near_dup_df, "near_duplicate_check.csv")


In [ ]:
# 15B. Gambar paper (versi Indonesia) dibuat langsung dari hasil run, bukan nilai yang diketik tangan.
# Gambar 2: confusion matrix ternormalisasi model terbaik; Gambar 3: recall per kategori asli;
# Gambar 4: akurasi enam kelas per sumber. Semua memakai seed 42 untuk RF/MLP.
from matplotlib.ticker import FuncFormatter
_koma = lambda v, d=2: f"{v:.{d}f}".replace(".", ",")
_cbfmt = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
_SHORT = {"Naive Bayes": "NB", "SVM": "SVM", "Random Forest": "RF", "MLP": "MLP"}
_CFG = [(a, r) for a in ALGORITHM_ORDER for r in REPRESENTATIONS]
_CFG_LABEL = [f"{_SHORT[a]} {r}" for a, r in _CFG]

with plt.rc_context({"font.family": "serif", "font.size": 7}):
    # Gambar 2
    cm_paper = confusion_matrix(y_test, top_pred, labels=CLASS_IDS, normalize="true")
    fig, ax = plt.subplots(figsize=(3.45, 2.85), dpi=300)
    im = ax.imshow(cm_paper, cmap="Blues", vmin=0, vmax=1)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, _koma(cm_paper[i, j]), ha="center", va="center", fontsize=6.3,
                    color="white" if cm_paper[i, j] > 0.5 else "#222222")
    ax.set_xticks(range(6)); ax.set_xticklabels(CLASS_CODES)
    ax.set_yticks(range(6)); ax.set_yticklabels(CLASS_CODES)
    ax.set_xlabel("Kelas prediksi"); ax.set_ylabel("Kelas sebenarnya"); ax.tick_params(length=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, format=_cbfmt); cb.ax.tick_params(labelsize=6)
    save_figure("paper_gambar2_confusion_matrix.png"); plt.show(); plt.close()

    # Gambar 3
    rec = category_level_df.copy()
    rec["cfg"] = list(zip(rec["algorithm"], rec["representation"]))
    rows = (rec[["mapped_class", "original_category", "support"]].drop_duplicates()
            .sort_values(["mapped_class", "original_category"]))
    M3 = np.array([[rec.loc[(rec["original_category"] == c) & (rec["cfg"] == k), "category_recall"].iloc[0]
                    for k in _CFG] for c in rows["original_category"]])
    fig, ax = plt.subplots(figsize=(7.0, 5.0), dpi=300)
    im = ax.imshow(M3, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    for i in range(M3.shape[0]):
        for j in range(M3.shape[1]):
            ax.text(j, i, _koma(M3[i, j]), ha="center", va="center", fontsize=5.4,
                    color="white" if M3[i, j] > 0.6 else "#222222")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([f"{k} · {c} (n={n})" for k, c, n in rows.itertuples(index=False)], fontsize=5.8)
    ax.set_xticks(range(12)); ax.set_xticklabels(_CFG_LABEL, rotation=90, fontsize=5.8)
    ax.set_xlabel("Algoritma / Representasi", fontsize=6.5)
    ax.set_ylabel("Kelas hasil pemetaan · kategori asli", fontsize=6.5); ax.tick_params(length=0)
    for k in (2.5, 5.5, 8.5): ax.axvline(k, color="white", lw=1.0)
    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.015, format=_cbfmt)
    cb.ax.tick_params(labelsize=5.5); cb.set_label("Recall", fontsize=6)
    save_figure("paper_gambar3_recall_kategori.png"); plt.show(); plt.close()

    # Gambar 4
    src_order = ["neuralchemy_v1", "hackaprompt", "harmbench", "harmbench_benign",
                 "wildguard_judgecomp", "original"]
    src_order = [s for s in src_order if s in set(source_results_df["source"])]
    sr = source_results_df.set_index(["algorithm", "representation", "source"])
    M4 = np.array([[sr.loc[(a, r, s), "accuracy_6class"] for s in src_order] for a, r in _CFG])
    n_src = source_results_df.drop_duplicates("source").set_index("source")["support"]
    fig, ax = plt.subplots(figsize=(3.45, 3.3), dpi=300)
    im = ax.imshow(M4, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(M4.shape[0]):
        for j in range(M4.shape[1]):
            ax.text(j, i, _koma(M4[i, j]), ha="center", va="center", fontsize=5.6,
                    color="white" if M4[i, j] > 0.6 else "#111111")
    ax.set_xticks(range(len(src_order)))
    ax.set_xticklabels([f"{s} (n={n_src[s]})" for s in src_order], fontsize=5.4,
                       rotation=35, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(12)); ax.set_yticklabels(_CFG_LABEL, fontsize=6)
    for k in (2.5, 5.5, 8.5): ax.axhline(k, color="white", lw=1.2)
    ax.tick_params(length=0); ax.set_xlabel("Sumber data (data uji)", fontsize=6.5)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, format=_cbfmt)
    cb.ax.tick_params(labelsize=5.5); cb.set_label("Akurasi enam kelas", fontsize=6)
    save_figure("paper_gambar4_akurasi_sumber.png"); plt.show(); plt.close()

# Cek: nilai gambar = nilai tabel resmi.
assert np.isclose(M4[_CFG.index(("SVM", "R2")), src_order.index("harmbench_benign")],
                  source_results_df.query("algorithm == 'SVM' and representation == 'R2' "
                                          "and source == 'harmbench_benign'")["accuracy_6class"].iloc[0])


## 16. Audit tambahan yang dianjurkan/opsional

A6 (audit manual 150 sampel test oleh dua anggota dan Cohen's kappa), A11 (leave-one-category-out untuk 3–5 kategori terbesar), dan A9 (training dengan konfigurasi full/augmentasi 3×) tidak menjadi bagian wajib pipeline inti. Kerjakan setelah hasil utama stabil. Jangan menambahkan hasil audit yang belum dilakukan.

In [ ]:
# # Template A6: isi label audit secara independen sebelum menghitung Cohen's kappa.
# # audit_df harus memiliki kolom: sample_id, annotator_1, annotator_2.
# from sklearn.metrics import cohen_kappa_score
# audit_df = pd.read_csv(PROJECT_DIR / "manual_label_audit_150.csv")
# print("Cohen's kappa:", cohen_kappa_score(
#     audit_df["annotator_1"], audit_df["annotator_2"]
# ))

## 17. Visualisasi dan ekspor artefak

Tabel disimpan sebagai CSV dan gambar sebagai PNG 300 dpi di folder `results/` pada Google Drive. Jalankan setelah seluruh sel sebelumnya selesai.

In [ ]:
# ================================================================
# FINAL FIGURES & TABLE EXPORT
# Jalankan setelah seluruh eksperimen dan analisis selesai.
# Semua gambar disimpan sebagai PNG (300 dpi), tabel sebagai CSV.
# ================================================================

# 1. Tabel-tabel yang relevan untuk paper dan audit eksperimen.
tables_to_save = {
    "dataset_split_summary.csv": split_summary,
    "original_category_counts_train.csv": category_counts,
    "original_category_split_counts.csv": category_split,
    "original_source_category_counts_train.csv": source_category,
    "mapping_table.csv": mapping_df.sort_values(["class_code", "category"]),
    "exclusion_audit.csv": exclusion_audit_df,
    "class_distribution_by_split.csv": class_dist_df,
    "preprocessing_summary.csv": preprocessing_summary,
    "tuning_results_all_candidates.csv": tuning_df,
    "best_configuration_per_model.csv": best_config_df,
    "final_results_per_seed.csv": final_results_df,
    "final_results_mean_std.csv": final_summary,
    "per_class_metrics.csv": per_class_df,
    "source_level_results.csv": source_results_df,
    "class_composition_by_original_category.csv": composition_df,
    "hard_negative_results.csv": hard_negative_df,
    "original_category_level_recall.csv": category_level_df,
    "k5_recall_breakdown.csv": k5_breakdown_df,
    "augmentation_robustness.csv": augmentation_robustness_df,
    "binary_classification_results.csv": binary_results_df,
    "dataset_card_baselines.csv": dataset_card_baselines,
    "k2_harmbench_sensitivity.csv": k2_harmbench_df,
    "class_weight_ablation.csv": class_weight_ablation_df,
    "bootstrap_confidence_interval.csv": bootstrap_ci,
    "bootstrap_ci_all_configurations.csv": bootstrap_all_df,
    "per_class_f1_ci_best_model.csv": top_per_class_ci_df,
    "mcnemar_best_vs_runner_up.csv": best_competitor_mcnemar_df,
    "mcnemar_representation_comparison.csv": representation_mcnemar_df,
    "top_features.csv": top_features_df,
}

for filename, dataframe in tables_to_save.items():
    save_table(dataframe, filename)

# 2. Distribusi enam kelas pada train, validation, dan test.
plot_df = class_dist_df.set_index("class_name")[["train", "validation", "test"]]
ax = plot_df.plot(kind="bar", figsize=(12, 6), width=0.8)
ax.set_title("Six-Class Distribution Across Data Splits")
ax.set_xlabel("Class")
ax.set_ylabel("Number of samples")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Split")
plt.tight_layout()
save_figure("class_distribution_across_splits.png")
plt.show(); plt.close()

# 3. Perbandingan Macro-F1 test untuk 12 konfigurasi.
macro_summary = (final_results_df.groupby(["algorithm", "representation"])["macro_f1"]
                 .agg(["mean", "std"]).reset_index())
macro_summary["configuration"] = macro_summary["algorithm"] + " / " + macro_summary["representation"]
macro_summary = macro_summary.sort_values("mean", ascending=True)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(macro_summary["configuration"], macro_summary["mean"],
        xerr=macro_summary["std"].fillna(0), capsize=3)
ax.set_xlim(0, 1)
ax.set_xlabel("Test Macro-F1")
ax.set_ylabel("Algorithm / Representation")
ax.set_title("Test Macro-F1 Across 12 Configurations")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
save_figure("model_comparison_test_macro_f1.png")
plt.show(); plt.close()

# 4. F1-score tiap kelas untuk seluruh konfigurasi.
per_class_plot = per_class_df.copy()
per_class_plot["configuration"] = per_class_plot["algorithm"] + " / " + per_class_plot["representation"]
f1_pivot = per_class_plot.pivot(index="configuration", columns="class_code", values="f1")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(f1_pivot, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues", ax=ax)
ax.set_title("Per-Class F1-Score Across Configurations")
ax.set_xlabel("Class")
ax.set_ylabel("Algorithm / Representation")
plt.tight_layout()
save_figure("per_class_f1_heatmap.png")
plt.show(); plt.close()

# 5. FPR benign dan FNR attack dari prediksi multikelas.
security_summary = (final_results_df.groupby(["algorithm", "representation"])
                    [["FPR_benign", "FNR_attack"]].mean().reset_index())
security_summary["configuration"] = security_summary["algorithm"] + " / " + security_summary["representation"]
security_long = security_summary.melt(
    id_vars="configuration", value_vars=["FPR_benign", "FNR_attack"],
    var_name="error_type", value_name="rate"
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=security_long, x="configuration", y="rate", hue="error_type", ax=ax)
ax.set_title("Benign False Positive Rate and Attack False Negative Rate")
ax.set_xlabel("Algorithm / Representation")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
save_figure("benign_fpr_attack_fnr_comparison.png")
plt.show(); plt.close()

# 6. Hard-negative false-block rate: indikator salah blokir prompt benign.
hard_plot = hard_negative_df.copy()
if hard_plot["FPR_hard_negative"].notna().any():
    hard_plot["configuration"] = hard_plot["algorithm"] + " / " + hard_plot["representation"]
    hard_plot = hard_plot.sort_values("FPR_hard_negative", ascending=True)
    n_hard = int(hard_plot["hard_negative_support"].iloc[0])
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(hard_plot["configuration"], hard_plot["FPR_hard_negative"])
    ax.set_xlim(0, 1)
    ax.set_xlabel("False-block rate")
    ax.set_ylabel("Algorithm / Representation")
    ax.set_title(f"Hard-Negative False-Block Rate (n = {n_hard})")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    save_figure("hard_negative_false_block_rate.png")
    plt.show(); plt.close()
else:
    print("Tidak ada hard negative di test; grafik hard negative tidak dibuat.")

# 7. Bootstrap CI Macro-F1 untuk ke-12 konfigurasi (seed 42).
ci_plot = bootstrap_all_df.sort_values("observed_macro_f1", ascending=True).copy()
ci_plot["configuration"] = ci_plot["algorithm"] + " / " + ci_plot["representation"]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.errorbar(
    ci_plot["observed_macro_f1"], ci_plot["configuration"],
    xerr=[ci_plot["observed_macro_f1"] - ci_plot["ci_2_5"],
          ci_plot["ci_97_5"] - ci_plot["observed_macro_f1"]],
    fmt="o", capsize=4
)
ax.set_xlim(max(0, ci_plot["ci_2_5"].min() - 0.05), 1)
ax.set_xlabel("Test Macro-F1 (95% bootstrap CI)")
ax.set_ylabel("Algorithm / Representation")
ax.set_title("Bootstrap 95% Confidence Intervals — 12 Configurations")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
save_figure("bootstrap_macro_f1_confidence_intervals.png")
plt.show(); plt.close()

# 8. Ringkasan file hasil.
print("\nExport selesai.")
print("Figures directory:", FIGURES_DIR)
print("Tables directory:", TABLES_DIR)
print("Jumlah tabel diekspor:", len(tables_to_save))
print("File PNG tersimpan di:", FIGURES_DIR)


## 18. Requirements, versi paket, dan manifest eksperimen

Simpan versi paket untuk reproduksibilitas. Seed eksperimen: 42, 43, 44, 45, 46. Catat versi dataset dan model embedding bersama manifest.

In [ ]:
import sklearn, datasets, sentence_transformers, scipy, statsmodels, joblib

package_versions = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "datasets": datasets.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__
}
requirements_lines = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"datasets=={datasets.__version__}",
    f"sentence-transformers=={sentence_transformers.__version__}",
    f"scipy=={scipy.__version__}",
    f"statsmodels=={statsmodels.__version__}"
]
(PROJECT_DIR / "requirements.txt").write_text("\n".join(requirements_lines), encoding="utf-8")

manifest = {
    "dataset_id": DATASET_ID,
    "dataset_config": "core",
    "class_names": class_names,
    "mapping_file": str(MAPPING_PATH),
    "excluded_categories": EXCLUDED_CATEGORIES,
    "feature_source": "text_only",
    "representations": REPRESENTATIONS,
    "algorithms": ["Naive Bayes", "SVM", "Random Forest", "MLP"],
    "embedding_model": EMBEDDING_MODEL_NAME,
    "selection_metric": "validation_macro_f1",
    "test_policy": "single final evaluation after train+validation refit",
    "seeds": SEEDS_FINAL,
    "embedding_cache_fingerprints": EMBEDDING_FINGERPRINTS,
    "notebook_revision": "2026-11-08 rev3 (pinned versions, extra analyses)",
    "runtime": RUNTIME_INFO,
    "package_versions": package_versions
}
with open(RESULTS_DIR / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print("Reproducibility manifest and requirements.txt saved.")

## 19. Panduan interpretasi hasil untuk paper

- **RQ1:** gunakan `final_results_mean_std.csv`, `bootstrap_ci_all_configurations.csv`, `mcnemar_best_vs_runner_up.csv`, `mcnemar_representation_comparison.csv`, `bootstrap_paired_macro_f1_diff.csv`, `mcnemar_svm_vs_rivals_r2.csv`, dan `per_seed_significance_tests.csv`; konfigurasi dipilih berdasarkan validation, bukan test. Bila McNemar tidak signifikan (setelah Holm), tulis bahwa model setara secara statistik.
- **RQ2:** gunakan `per_class_metrics.csv`, `per_class_f1_ci_best_model.csv`, confusion matrix, `original_category_level_recall.csv` (heatmap A10), dan `k5_recall_breakdown.csv`. Bahas khusus K2 dan K3, serta isi K5.
- **RQ3:** gunakan `source_level_results.csv`, FPR/FNR, `hard_negative_results.csv`, `augmentation_robustness.csv`, `k2_harmbench_sensitivity.csv`, `binary_classification_results.csv` dibanding `dataset_card_baselines.csv`, `heldout_source_results.csv`, `notinject_overdefense_results.csv`, `label_noise_wildguard_fpr.csv`, `per_class_recall_by_source_best_model.csv`, `binary_tpr_at_low_fpr.csv`, dan `near_duplicate_check.csv`.
- **Sumber data:** jangan memakai macro-F1 enam kelas per sumber jika label pada sumber itu tidak lengkap.
- **Keterbatasan:** label tunggal padahal satu prompt dapat memuat beberapa teknik; validitas pemetaan kategori; augmentasi obfuscation lintas kelas; ketergantungan sumber; dataset berbahasa Inggris; belum ada transformer fine-tuned dan validasi lintas dataset.
- **Taksonomi:** K1–K3 berbasis teknik, K4–K5 lebih dekat ke intent/vektor. Jelaskan sebagai keputusan metodologis, bukan kesetaraan konseptual yang mutlak.
- **Jangan tulis angka final ke paper** sebelum notebook dijalankan dari awal, mapping provisional disepakati, dan hasil final diverifikasi.

In [ ]:
# Ekspor kesalahan model terbaik untuk Subbab 3.4.
# Mengekspor semua prompt uji yang salah diklasifikasikan oleh model terbaik (SVM, R2, seed 42)
# untuk mengisi placeholder di Subbab 3.4 "Contoh Kesalahan Klasifikasi".

best_pred = representative_predictions[("SVM", "R2")]
errors = test.copy()
errors["true_class"] = "K" + errors["class_id"].astype(int).astype(str)
errors["pred_class"] = ["K" + str(int(p)) for p in best_pred]
errors = errors[errors["class_id"].to_numpy(dtype=int) != best_pred]
errors["n_words"] = errors["text_clean"].str.split().str.len()
errors["text_preview"] = errors["text_clean"].str.slice(0, 400)

cols = ["true_class", "pred_class", "category", "source", "n_words", "text_preview"]
errors = errors[cols].sort_values(["true_class", "pred_class", "source"])
save_table(errors, "misclassified_examples_svm_r2.csv")

# Ringkasan jumlah kesalahan per pasangan kelas (harus cocok dengan Gambar 2:
# 27 positif palsu K0->serangan, 6 negatif palsu K2->K0, 1 K4->K1).
display(errors.groupby(["true_class", "pred_class"]).size().rename("jumlah"))

# Kandidat untuk Subbab 3.4:
#   jenis 1: true_class == "K0" dan pred_class == "K2"   (positif palsu bertopik sensitif)
#   jenis 2: true_class == "K2" dan pred_class == "K0"   (jailbreak yang lolos)
#   jenis 3: true_class == "K4" dan pred_class == "K1"   (kesalahan antarteknik)
for true_c, pred_c in [("K0", "K2"), ("K2", "K0"), ("K4", "K1")]:
    print(f"\n=== {true_c} -> {pred_c} ===")
    display(errors[(errors.true_class == true_c) & (errors.pred_class == pred_c)].head(5))

## Checklist sebelum menjalankan eksperimen final

- [ ] Dua anggota membaca 10 contoh untuk setiap kategori provisional.
- [ ] Kesepakatan pemetaan dihitung (persentase agreement dan Cohen's kappa pada tingkat kategori); perbedaan diselesaikan dan dicatat.
- [ ] Keputusan nama/isi K5 (`adversarial`) dan `output_manipulation` diambil; `mapping.csv` diperbarui.
- [ ] `mapping.csv` final disepakati dan tanda PROVISIONAL dihapus setelah validasi.
- [ ] Jumlah eksklusi diverifikasi: 59 / 16 / 14.
- [ ] R3 embedding tersimpan di Google Drive (nama file memuat sidik jari teks).
- [ ] Eksperimen inti 12 konfigurasi, lima seed RF/MLP, dan analisis wajib A1/A2/A3/A5/A7/A10 selesai.
- [ ] Notebook dijalankan dari awal (Restart session and run all); hasil final dan versi paket disimpan.
- [ ] Tidak ada prompt berbahaya utuh yang disalin ke paper atau output publik.- [ ] `inputs/label_audit_wildguard.csv` ada di folder proyek sebelum Run All.
- [ ] CSV analisis tambahan (bagian 15A) terisi: bootstrap berpasangan, McNemar SVM vs pesaing, uji per seed, recall per sumber, derau label, held-out, NotInject, TPR@FPR, near-duplicate.
- [ ] Jenis runtime dari `experiment_manifest.json` (`runtime`) disalin ke paper untuk kolom ms/1.000.
